# 1.Importăm librăriile de care avem nevoie 

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 2.Încărcare și curățare

In [9]:
df = pd.read_csv('../data/products.csv')
df.columns = df.columns.str.strip() # Eliminăm spațiile din numele coloanelor
df = df.dropna(subset=['Product Title', 'Category Label']).copy()

# 3. Feature Engineering

In [10]:
def apply_feature_engineering(data):
    data['char_count'] = data['Product Title'].apply(len)
    data['word_count'] = data['Product Title'].apply(lambda x: len(x.split()))
    data['digit_count'] = data['Product Title'].apply(lambda x: sum(c.isdigit() for c in x))
    return data

df = apply_feature_engineering(df)

# 4. Pregătirea datelor

In [11]:
X = df[['Product Title', 'char_count', 'word_count', 'digit_count']]
y = df['Category Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Pipeline-ul Modelului

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(ngram_range=(1,2), max_features=10000), 'Product Title'),
        ('num', StandardScaler(), ['char_count', 'word_count', 'digit_count'])
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LinearSVC(random_state=42, dual='auto'))
])

# 6. Antrenament şi evaluare

In [14]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"Acuratețe: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred,zero_division=0))

Acuratețe: 0.9563
                  precision    recall  f1-score   support

             CPU       0.00      0.00      0.00        17
            CPUs       0.98      1.00      0.99       749
 Digital Cameras       1.00      0.99      1.00       538
     Dishwashers       0.94      0.95      0.95       681
        Freezers       0.98      0.93      0.95       440
 Fridge Freezers       0.94      0.95      0.94      1094
         Fridges       0.88      0.91      0.89       687
      Microwaves       0.98      0.96      0.97       466
    Mobile Phone       0.00      0.00      0.00        11
   Mobile Phones       0.97      0.99      0.98       801
             TVs       0.98      0.99      0.99       708
Washing Machines       0.95      0.95      0.95       803
          fridge       0.00      0.00      0.00        25

        accuracy                           0.96      7020
       macro avg       0.74      0.74      0.74      7020
    weighted avg       0.95      0.96      0.95     